In [0]:
import time
start = time.time()

In [0]:
from pyspark.sql.functions import sum, round


# Read from silver
silver_df = spark.table('project_catalog.medallion_project.silver_table')

gold_df = silver_df.groupby(
  'Region',
  'Product_Category',
  'Customer_Type',
  'Sales_Channel'
).agg(
  round(sum('Sales_Amount'),2).alias('Total_Sales'),
  sum('Quantity_Sold').alias('Total_Quantity')
)

# Save gold
gold_df.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable('project_catalog.medallion_project.gold_table')

In [0]:
spark.sql("OPTIMIZE project_catalog.medallion_project.gold_table")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
%sql
select * from project_catalog.medallion_project.gold_table;

Region,Product_Category,Customer_Type,Sales_Channel,Total_Sales,Total_Quantity
East,Electronics,Returning,Online,67682.27,379
North,Clothing,Returning,Online,74127.35,388
West,Electronics,Returning,Retail,67505.48,232
West,Electronics,New,Retail,65194.93,337
North,Food,Returning,Online,50732.69,212
East,Electronics,New,Retail,55444.95,388
North,Furniture,New,Online,103140.71,548
West,Electronics,Returning,Online,79691.12,405
East,Food,New,Retail,98902.91,414
South,Furniture,Returning,Online,100048.4,456


In [0]:
import time


gold_df = spark.table('project_catalog.medallion_project.gold_table')

row_count = gold_df.count()

end = time.time()

print("=== Gold Metrics ===")
print("Rows:", row_count)
print("Execution Time (sec):", end - start)

=== Gold Metrics ===
Rows: 64
Execution Time (sec): 20.385616540908813


In [0]:
bronze_count = spark.table('project_catalog.medallion_project.bronze_table').count()
silver_count = spark.table('project_catalog.medallion_project.silver_table').count()
gold_count = spark.table('project_catalog.medallion_project.gold_table').count()

print("=== Data Movement ===")
print("Bronze → Silver:", bronze_count, "→", silver_count)
print("Silver → Gold:", silver_count, "→", gold_count)

=== Data Movement ===
Bronze → Silver: 1000 → 1000
Silver → Gold: 1000 → 64


In [0]:
def get_table_size(table_name):
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}")
    return detail.select("sizeInBytes").collect()[0][0] / (1024*1024)

bronze_size = get_table_size("project_catalog.medallion_project.bronze_table")
silver_size = get_table_size("project_catalog.medallion_project.silver_table")
gold_size = get_table_size("project_catalog.medallion_project.gold_table")

print("=== Data Size (MB) ===")
print("Bronze:", bronze_size)
print("Silver:", silver_size)
print("Gold:", gold_size)

=== Data Size (MB) ===
Bronze: 0.022355079650878906
Silver: 0.02232837677001953
Gold: 0.0025472640991210938
